In [ ]:
import os, subprocess, sys
from pathlib import Path
import pandas as pd

# Folder of this repository (it holds DDC_WINDOWS_Converters and ddc-tools)
REPO = Path(os.environ.get("DDC_REPO", r"C:\DDC\cad2data-Revit-IFC-DWG"))
sys.path.insert(0, str(REPO / "ddc-tools" / "src"))   # or install once:  pip install "<repo>\ddc-tools"
from ddc_tools import table

# DDC RVT Converter command line (folder DDC_WINDOWS_Converters/DDC_RVT_Converter, see its README.md)
rvt_cli = str(REPO / "DDC_WINDOWS_Converters" / "DDC_RVT_Converter" / "cli" / "RVTconverter.exe")
# The address of the Revit file on computer
file_path = os.environ.get("DDC_RVT", r"D:\RVT_projects\Project.rvt")
# Result folder (the converter refuses a folder that already exists)
out_dir = file_path[:-4] + ".converted"

# Convert a Revit project to open formats: GLB model, summary.json, report.json,
# properties.jsonl (--properties) and native_graph.ifc (--ifc: category, family, type and level names)
if not os.path.exists(os.path.join(out_dir, "properties.jsonl")):
    process = subprocess.run([rvt_cli, file_path, "-o", out_dir, "--properties", "--ifc"])
    print("converter exit code:", process.returncode)      # 0 = written, 2 = written with reported gaps, 1 = failed

In [ ]:
# Build the flat table with the columns of the old *_rvt.xlsx (ID, Category, Type Name, Level, Volume, ...)
# the same as the command line:  ddc table <out_dir> --xlsx <file>
output_file = file_path[:-4] + "_rvt.xlsx"
table.write_xlsx(table.build_table(out_dir), output_file)

# Read the table (sheet Project_Data, as in the old files)
df = pd.read_excel(output_file, sheet_name="Project_Data")
df.head(5)

In [ ]:
# Remove the storage type from parameter names ("Name : String" -> "Name"); the new tables have none, kept for old files
df.columns = [col.split(' : ')[0] for col in df.columns]
QTO_group = df.groupby('Type Name')['Volume'].sum().reset_index().query("Volume != 0")
QTO_group

In [ ]:
# Save the filter data to a new Excel file
QTO_group.to_excel(os.path.join(os.path.dirname(file_path), "QTO_group.xlsx"))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize = (10,12))
plt.barh(QTO_group["Type Name"], QTO_group["Volume"], color = "skyblue")
plt.xlabel('Volume'); plt.ylabel('Type Name'); plt.title('Volume by Type Name'); plt.tight_layout()